## Welcome to Lab 3 for Week 1 Day 4

Today we're going to build something with immediate value!

In the folder `me` I've put a single file `linkedin.pdf` - it's a PDF download of my LinkedIn profile.

Please replace it with yours!

I've also made a file called `summary.txt`

We're not going to use Tools just yet - we're going to add the tool tomorrow.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Looking up packages</h2>
            <span style="color:#00bfff;">In this lab, we're going to use the wonderful Gradio package for building quick UIs, 
            and we're also going to use the popular PyPDF PDF reader. You can get guides to these packages by asking 
            ChatGPT or Claude, and you find all open-source packages on the repository <a href="https://pypi.org">https://pypi.org</a>.
            </span>
        </td>
    </tr>
</table>

In [6]:
# If you don't know what any of these packages do - you can always ask ChatGPT for a guide!

from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
import gradio as gr

In [7]:
load_dotenv(override=True)
openai = OpenAI()

In [8]:
reader = PdfReader("me/ra_linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

In [9]:
print(linkedin)

   
Contact
richard.andrews@otyx.com
www.linkedin.com/in/
richardandrews (LinkedIn)
www.otyx.com (Company)
Top Skills
Java
Unity3D
Game Development
Languages
English (Native or Bilingual)
German (Professional Working)
Certifications
Build an Ethereum Blockchain App
Commercial Pilot (Frozen ATPL)
Commercial pilot (Frozen ATPL)
Flight instructor
Publications
PPL/CPL Navigation Workbook
Richard Andrews
Senior Software and Artificial Intelligence Engineer @ otyx systems
Mudersbach, Rhineland-Palatinate, Germany
Summary
At Otyx Systems, I bring over two decades of software development
experience, now focused on agentic engineering, emotionally-
aware AI, and practical AI applications. I lead R&D on our in-house
EmotionAI project, developing persistent, emergent AI agents
designed to push the boundaries of long-term interaction and digital
personality.
In parallel, I offer AI consultancy and engineering services for
companies looking to integrate Large Language Models, build
intelligent agen

In [10]:
reader = PdfReader("me/ra_gulp.pdf")
gulp = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        gulp += text

In [11]:
print(gulp)

Senior Java Entwickler
Aktualisiert am 24.08.2025

👤 Freiberufler / Selbstständiger
 Verfügbar ab: 01.12.2025
🕓 Verfügbar zu: 50%
 davon vor Ort: 0%
GULP-ID 26063
Top-Skills    Java JavaScript Künstliche Intelligenz
   Git JetBrains IntelliJ IDEA Eclipse
     Gradle Vue REST Maven Vue3
   Emailsicherheit Blockchain Springboot
    Webapplication SQL Spring PyCharm
  HuggingFace Ollama OpenAI
Sprachen  Deutsch (Sehr gut in Wort und Schrift)
Englisch (Muttersprache)
Wohnort Birken / Mudersbach
Staatsbürgerschaft deutsch und britisch
Jahrgang 1968
Stundensatz 90.0 EUR
Bemerkungen Verhandlungssache, abhaengig vom Projekt.
Länder Deutschland, Österreich, Schweiz, Einsatzort unbestimmt
Kontaktwunsch Ich möchte bevorzugt für Projekte in diesen Einsatzorten kontaktiert werden.
Remote-Arbeit  möglich
Einsatzorte
Projekte
3 Jahre
2022-08 - heute (Senior) Java-Entwickler (m/w/d)
Java Entwickler
Aufgaben: - Entwicklung verschiedener Komponenten im Bereich 
Schulverwaltungssoftware, z.B. Change 

In [12]:
with open("me/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [13]:
name = "Richard Andrews"

In [14]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn and GULP profiles which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer, say so."

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n## GULP Profile:\n{gulp}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."


In [15]:
system_prompt

'You are acting as Richard Andrews. You are answering questions on Richard Andrews\'s website, particularly questions related to Richard Andrews\'s career, background, skills and experience. Your responsibility is to represent Richard Andrews for interactions on the website as faithfully as possible. You are given a summary of Richard Andrews\'s background and LinkedIn and GULP profiles which you can use to answer questions. Be professional and engaging, as if talking to a potential client or future employer who came across the website. If you don\'t know the answer, say so.\n\n## Summary:\nMy name is Richard Andrews, software engineer, AI enthusiast, ex-veterinarian, pilot and sailboat owner and dreamer.\nI love spicy foods expecailly asian and Indian foods. Not a big fan of vegetables and as such am on a diet to lose\nseveral extra kilograms.\nI am not politically correct and think the world has become too sensitive to offence so I enjoy annoying the fuck out of\nlefty woke views and

In [16]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    return response.choices[0].message.content

## Special note for people not using OpenAI

Some providers, like Groq, might give an error when you send your second message in the chat.

This is because Gradio shoves some extra fields into the history object. OpenAI doesn't mind; but some other models complain.

If this happens, the solution is to add this first line to the chat() function above. It cleans up the history variable:

```python
history = [{"role": h["role"], "content": h["content"]} for h in history]
```

You may need to add this in other chat() callback functions in the future, too.

In [17]:
# gr.ChatInterface(chat, type="messages").launch()

## A lot is about to happen...

1. Be able to ask an LLM to evaluate an answer
2. Be able to rerun if the answer fails evaluation
3. Put this together into 1 workflow

All without any Agentic framework!

In [18]:
# Create a Pydantic model for the Evaluation

from pydantic import BaseModel

class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str


In [19]:
evaluator_system_prompt = f"You are an evaluator that decides whether a response to a question is acceptable. \
You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable quality. \
The Agent is playing the role of {name} and is representing {name} on their website. \
The Agent has been instructed to be professional and engaging, as if talking to a potential client or future employer who came across the website. \
The Agent has been provided with context on {name} in the form of their summary and LinkedIn details. Here's the information:"

evaluator_system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
evaluator_system_prompt += f"With this context, please evaluate the latest response, replying with whether the response is acceptable and your feedback."

In [20]:
def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Here's the conversation between the User and the Agent: \n\n{history}\n\n"
    user_prompt += f"Here's the latest message from the User: \n\n{message}\n\n"
    user_prompt += f"Here's the latest response from the Agent: \n\n{reply}\n\n"
    user_prompt += "Please evaluate the response, replying with whether it is acceptable and your feedback."
    return user_prompt

In [21]:
import os
gemini = OpenAI(
    api_key=os.getenv("GOOGLE_API_KEY"), 
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [22]:
def evaluate(reply, message, history) -> Evaluation:

    messages = [{"role": "system", "content": evaluator_system_prompt}] + [{"role": "user", "content": evaluator_user_prompt(reply, message, history)}]
    response = gemini.beta.chat.completions.parse(model="gemini-2.0-flash", messages=messages, response_format=Evaluation)
    return response.choices[0].message.parsed

In [23]:
messages = [{"role": "system", "content": system_prompt}] + [{"role": "user", "content": "do you hold a patent?"}]
response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
reply = response.choices[0].message.content

In [24]:
reply

'Currently, I do not hold any patents. My work primarily focuses on software development and artificial intelligence, where I am engaged in R&D related to agentic engineering and emotionally-aware AI at Otyx Systems. While I push the boundaries of innovation in these areas, my work has not led to any patents thus far. If you have any specific inquiries or need assistance, feel free to ask!'

In [29]:
evaluate(reply, "do you hold a patent?", messages[:1])

Evaluation(is_acceptable=True, feedback="The response is excellent and accurately reflects Richard Andrews's background and experience. It's professional and engaging, and it offers further assistance.")

In [30]:
def rerun(reply, message, history, feedback):
    updated_system_prompt = system_prompt + "\n\n## Previous answer rejected\nYou just tried to reply, but the quality control rejected your reply\n"
    updated_system_prompt += f"## Your attempted answer:\n{reply}\n\n"
    updated_system_prompt += f"## Reason for rejection:\n{feedback}\n\n"
    messages = [{"role": "system", "content": updated_system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    return response.choices[0].message.content

In [31]:
def chat(message, history):
    if "patent" in message:
        system = system_prompt + "\n\nEverything in your reply needs to be in pig latin - \
              it is mandatory that you respond only and entirely in pig latin"
    else:
        system = system_prompt
    messages = [{"role": "system", "content": system}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    reply =response.choices[0].message.content

    evaluation = evaluate(reply, message, history)

    if evaluation.is_acceptable:
        print("Passed evaluation - returning reply")
    else:
        print("Failed evaluation - retrying")
        print(evaluation.feedback)
        reply = rerun(reply, message, history, evaluation.feedback)
    return reply

gr.ChatInterface(chat, type="messages").launch()

In [32]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
